# Step 3: Sentiment / Emotion Analysis

This notebook extracts **emotional tone** from each book description so the dashboard can filter recommendations by mood (Happy, Sad, Suspenseful, etc.).

**Approach:**
1. Split each description into sentences.
2. Run a pretrained emotion classifier on every sentence.
3. Take the **maximum score per emotion** across all sentences — this captures the strongest emotional signal in the book.

Model: `j-hartmann/emotion-english-distilroberta-base` (7 emotions: anger, disgust, fear, joy, sadness, surprise, neutral).

**Pipeline position:**
```
books_with_categories.csv  →  [this notebook]  →  books_with_emotions.csv  →  Gradio dashboard
```

## Load data and set up the emotion classifier

In [ ]:
# Load categorized books and initialize emotion classifier.
import numpy as np
import pandas as pd
from transformers import pipeline

books = pd.read_csv("datasets/books_with_categories.csv")

classifier = pipeline(
    "text-classification",
    model="j-hartmann/emotion-english-distilroberta-base",
    top_k=None,
    device=-1,
)

## Aggregate sentence-level emotions to book-level scores

A single description may contain many sentences with different tones. We score each sentence, then keep the **peak score** for each emotion label — e.g. one joyful sentence is enough to mark a book as having high *joy*.

In [ ]:
# Define helper to compute max emotion scores across sentences.
emotion_labels = ["anger", "disgust", "fear", "joy", "sadness", "surprise", "neutral"]

def calculate_max_emotion_scores(predictions):
    per_emotion_scores = {label: [] for label in emotion_labels}
    for prediction in predictions:
        sorted_predictions = sorted(prediction, key=lambda x: x["label"])
        for index, label in enumerate(emotion_labels):
            per_emotion_scores[label].append(sorted_predictions[index]["score"])
    return {label: np.max(scores) for label, scores in per_emotion_scores.items()}

## Score all books and merge results

In [ ]:
# Compute emotion scores for every book.
from tqdm import tqdm

isbn = []
emotion_scores = {label: [] for label in emotion_labels}

for i in tqdm(range(len(books))):
    isbn.append(books["isbn13"][i])
    sentences = books["description"][i].split(".")
    predictions = classifier(sentences)
    max_scores = calculate_max_emotion_scores(predictions)
    for label in emotion_labels:
        emotion_scores[label].append(max_scores[label])

emotions_df = pd.DataFrame(emotion_scores)
emotions_df["isbn13"] = isbn

books = pd.merge(books, emotions_df, on="isbn13")
books.to_csv("datasets/books_with_emotions.csv", index=False)
print(f"Saved {len(books):,} books to datasets/books_with_emotions.csv")